# 03 — CRNN + Cross-Hive Generalization
**Month 2, Week 7–8 Deliverable — The Headline Result**

Goals:
1. Train the CRNN model
2. Compare CRNN vs Baseline CNN on within-hive accuracy
3. Run the cross-hive generalization experiment
4. Produce the within-hive vs unseen-hive accuracy table
5. Visualise the generalization gap — this is your strongest slide


In [ ]:
import sys
sys.path.insert(0, '..')

import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import config

print('Ready.')


## Step 1 — Train CRNN (run from CLI for full training)

In [ ]:
# Run this in the terminal (faster than running inside notebook):
#
#   python train.py --model crnn --epochs 50 --experiment exp_02 --cross-hive
#
# The --cross-hive flag runs cross_hive_report() after training automatically.

print('Run in terminal: python train.py --model crnn --epochs 50 --experiment exp_02 --cross-hive')


## Step 2 — Compare Baseline CNN vs CRNN

In [ ]:
from data import get_dataloaders
from models import get_model
from train import evaluate, get_device

device = get_device()
_, _, test_loader = get_dataloaders()

results = {}
for exp, model_name in [('exp_01', 'baseline_cnn'), ('exp_02', 'crnn')]:
    ckpt_path = os.path.join(config.CHECKPOINT_DIR, f'{exp}_best.pt')
    if not os.path.exists(ckpt_path):
        print(f'No checkpoint for {exp} — skipping')
        continue
    ckpt  = torch.load(ckpt_path, map_location=device)
    model = get_model(model_name).to(device)
    model.load_state_dict(ckpt['model_state'])
    m = evaluate(model, test_loader, device, split_name='within-hive test')
    results[model_name] = m

# Print comparison table
comparison = pd.DataFrame([
    {'Model': k, 'Accuracy': f"{v['accuracy']*100:.2f}%",
     'F1': f"{v['f1']*100:.2f}%", 'Precision': f"{v['precision']*100:.2f}%",
     'Recall': f"{v['recall']*100:.2f}%"}
    for k, v in results.items()
])
print('\nWithin-hive Test Set Comparison:')
print(comparison.to_string(index=False))


## Step 3 — Cross-Hive Generalization Report

In [ ]:
from train import cross_hive_report

# Load best CRNN
ckpt_path = os.path.join(config.CHECKPOINT_DIR, 'exp_02_best.pt')
if os.path.exists(ckpt_path):
    ckpt  = torch.load(ckpt_path, map_location=device)
    model = get_model('crnn').to(device)
    model.load_state_dict(ckpt['model_state'])
    
    report = cross_hive_report(model, device)
else:
    # Load from saved CSV if already run
    report_path = os.path.join(config.DATA_PROCESSED, 'cross_hive_report.csv')
    if os.path.exists(report_path):
        report = pd.read_csv(report_path)
        print(report)
    else:
        print('Train CRNN first.')


## Step 4 — Visualise the Generalization Gap (Your Strongest Slide)

In [ ]:
if report is not None and len(report) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    colors = {'train': '#22c55e', 'val': '#f59e0b', 'test': '#ef4444'}
    
    bars = ax.bar(
        report['hive_id'],
        report['accuracy'],
        color=[colors.get(s, '#888') for s in report['split']],
        edgecolor='white', linewidth=0.5
    )
    
    # Horizontal lines for averages
    train_avg = report[report['split']=='train']['accuracy'].mean()
    test_avg  = report[report['split']=='test']['accuracy'].mean()
    ax.axhline(train_avg, color='#22c55e', linestyle='--', alpha=0.7, label=f'Train avg: {train_avg:.1f}%')
    ax.axhline(test_avg,  color='#ef4444', linestyle='--', alpha=0.7, label=f'Test avg: {test_avg:.1f}%')
    
    # Annotate gap
    ax.annotate(
        f'Generalization gap: {train_avg - test_avg:.1f}%',
        xy=(0.98, (train_avg + test_avg) / 2),
        xycoords=('axes fraction', 'data'),
        fontsize=11, color='white', fontweight='bold',
        ha='right', va='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='#ef4444', alpha=0.8)
    )
    
    # Legend for split colours
    from matplotlib.patches import Patch
    legend_els = [Patch(fc=c, label=s.upper()) for s, c in colors.items()]
    ax.legend(handles=legend_els + ax.get_lines(), loc='lower right', framealpha=0.2)
    
    ax.set_xlabel('Hive ID', fontsize=12)
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title('CRNN — Per-Hive Accuracy\n(Green = Training Hives | Red = Unseen Test Hives)',
                 fontsize=13)
    ax.set_ylim(0, 110)
    ax.tick_params(axis='x', rotation=45)
    ax.set_facecolor('#1a1a1a')
    fig.patch.set_facecolor('#0d0d0f')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    
    plt.tight_layout()
    plt.savefig('cross_hive_generalization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved → cross_hive_generalization.png (use in your slide deck)')


## Month 2 Checkpoint Summary

| Metric | Baseline CNN | CRNN |
|---|---|---|
| Within-hive accuracy | ??? | ??? |
| Within-hive F1 | ??? | ??? |
| Cross-hive accuracy (test hives) | ??? | ??? |
| Generalization gap | ??? | ??? |

**The generalization gap is your headline result — present it directly and honestly.**
A gap of 10–20% is completely expected and is a meaningful finding, not a failure.
